# Okoliš AI — Trening RandLA-Net

8-klasna semantička segmentacija point cloudova (Toronto3D + SemanticKITTI).

**GPU:** RTX A6000 (48 GB VRAM)

## 1. Provjera GPU-a

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram:.1f} GB")
else:
    print("GRESKA: CUDA nije dostupan!")

## 2. Instalacija zavisnosti

In [ ]:
!pip install plyfile pyyaml scipy -q

## 3. Kloniranje repozitorija

In [ ]:
import os
os.chdir("/workspace")

REPO_URL = "https://github.com/TVOJ_USERNAME/okolis-training.git"  # <-- PROMIJENI OVO

if not os.path.exists("okolis-training"):
    !git clone {REPO_URL}
else:
    print("Repo vec postoji, pulling...")
    !cd okolis-training && git pull

os.chdir("/workspace/okolis-training")
!ls -la

## 4. Preuzimanje Toronto3D dataseta

Toronto3D je slobodno dostupan na Zenodo — ne zahtijeva registraciju.

In [ ]:
!mkdir -p /workspace/data/Toronto_3D
os.chdir("/workspace/data/Toronto_3D")

if not os.path.exists("L001.ply"):
    print("Preuzimanje Toronto3D (~1.5 GB)...")
    !wget -q --show-progress https://zenodo.org/records/4570104/files/Toronto_3D.zip
    !unzip -o Toronto_3D.zip
    !rm -f Toronto_3D.zip
else:
    print("Toronto3D vec preuzet.")

!ls -lh *.ply 2>/dev/null || echo "GRESKA: Nema .ply datoteka!"

## 5. Preuzimanje SemanticKITTI dataseta

SemanticKITTI zahtijeva **registraciju**.

### Koraci:
1. Otvori https://www.semantic-kitti.org/ u browseru (može na tvom lokalnom računalu)
2. Registriraj se i prihvati uvjete korištenja
3. Idi na http://www.cvlibs.net/datasets/kitti/eval_odometry.php
4. Preuzmi **"Download odometry data set (velodyne laser data, 80 GB)"**
5. Sa semantic-kitti.org preuzmi **label datoteke**
6. Uploadaj na VM ili koristi `wget` s linkovima

**Alternativa:** Koristi `gdown` ili `rclone` za prijenos s Google Drivea.

Kad imaš podatke, raspakiraj ih u strukturu ispod:

In [ ]:
!mkdir -p /workspace/data/SemanticKITTI/dataset/sequences

# === AKO VEC IMAS ZIP DATOTEKE, RASPAKIRAJ IH OVDJE ===
# Primjer (prilagodi nazive datoteka):
#
# !cd /workspace/data/SemanticKITTI && unzip data_odometry_velodyne.zip
# !cd /workspace/data/SemanticKITTI && unzip data_odometry_labels.zip
#
# === ILI PREUZMI DIREKTNO (zamijeni URL-ove) ===
# !cd /workspace/data/SemanticKITTI && wget "TVOJ_DOWNLOAD_LINK_VELODYNE"
# !cd /workspace/data/SemanticKITTI && wget "TVOJ_DOWNLOAD_LINK_LABELS"

print("Provjera strukture:")
!ls /workspace/data/SemanticKITTI/dataset/sequences/ 2>/dev/null || echo "UPOZORENJE: sequences/ folder ne postoji!"
!ls /workspace/data/SemanticKITTI/dataset/sequences/00/velodyne/ 2>/dev/null | head -3 || echo "UPOZORENJE: seq 00 nema velodyne podataka!"

## 6. Verifikacija dataseta

In [ ]:
import os

def check_toronto3d(root):
    files = [f for f in os.listdir(root) if f.endswith('.ply')]
    print(f"Toronto3D: {len(files)} PLY datoteka")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f"  {f}: {size_mb:.0f} MB")
    return len(files) > 0

def check_semantickitti(root):
    seq_dir = os.path.join(root, "dataset", "sequences")
    if not os.path.exists(seq_dir):
        seq_dir = os.path.join(root, "sequences")
    if not os.path.exists(seq_dir):
        print("SemanticKITTI: NIJE PRONADEN")
        return False
    seqs = sorted(os.listdir(seq_dir))
    total_bins = 0
    total_labels = 0
    for s in seqs:
        vel = os.path.join(seq_dir, s, "velodyne")
        lbl = os.path.join(seq_dir, s, "labels")
        nb = len(os.listdir(vel)) if os.path.exists(vel) else 0
        nl = len(os.listdir(lbl)) if os.path.exists(lbl) else 0
        total_bins += nb
        total_labels += nl
        print(f"  seq {s}: {nb} bins, {nl} labels")
    print(f"  Ukupno: {total_bins} bin, {total_labels} label")
    return total_bins > 0 and total_labels > 0

print("=" * 50)
ok1 = check_toronto3d("/workspace/data/Toronto_3D")
print("=" * 50)
ok2 = check_semantickitti("/workspace/data/SemanticKITTI")
print("=" * 50)

if ok1 and ok2:
    print("\nSVI DATASETI SPREMNI — mozete pokrenuti trening!")
elif ok1:
    print("\nToronto3D spreman. SemanticKITTI nedostaje (trening ce raditi samo s Toronto3D).")
else:
    print("\nGRESKA: Potreban je barem Toronto3D dataset!")

## 7. Pokretanje treninga

In [ ]:
os.chdir("/workspace/okolis-training")

# Provjeri RAM prije treninga
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"Dostupan RAM: {ram_gb:.0f} GB")
if ram_gb < 40:
    print("UPOZORENJE: Malo RAM-a! Povecaj kitti_scan_stride u config.yaml.")

print("\nPokretanje treninga...\n")
!python train.py --config config.yaml

## 8. Provjera rezultata

In [ ]:
import torch

ckpt_path = "/workspace/runs/rtx_a6000/best.pt"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    print(f"Best model:")
    print(f"  Epoch: {ckpt['epoch']}")
    print(f"  mIoU:  {ckpt['miou']:.4f}")
    print(f"  Loss:  {ckpt['loss']:.4f}")
    print(f"  Config: {ckpt['cfg']}")
    print(f"\nDatoteka: {ckpt_path}")
    print(f"Velicina: {os.path.getsize(ckpt_path)/1e6:.1f} MB")
    print("\nPreuzmi best.pt i koristi u glavnom projektu s test_scan.py!")
else:
    print("GRESKA: best.pt ne postoji. Trening jos nije zavrsen ili je crashao.")
    if os.path.exists("/workspace/runs/rtx_a6000/last.pt"):
        ckpt = torch.load("/workspace/runs/rtx_a6000/last.pt", map_location="cpu", weights_only=False)
        print(f"\nPostoji last.pt (epoch {ckpt['epoch']}, mIoU={ckpt['miou']:.4f})")